# 06 — Yearly Skill Forecasting (2027)

**Goal:** Run yearly time-series forecasting to predict skill demand share for the year 2027 based on 2020–2026 data.

## Methodology
Because only 7 annual data points are available, we apply a tiered approach to ensure honest modeling:
1. **Minimum Coverage:** Requires $\ge 5$ years of historical data points.
2. **Prophet Model:** Fit a yearly Prophet model (no sub-annual seasonality).
3. **Uncertainty Fallback:** Check the 80% confidence interval width. If the CI is wider than $1.5 \times \text{forecast}$, it falls back to a simple **linear regression** trend extrapolation.
4. **Trend Classification:** Compare the 2027 prediction to the 2026 actuals (change $> 10\%$ is Rising, $<-10\%$ is Declining, else Stable).

**Input:** `data/clean/primary_skills_long.csv`  
**Output:** `data/clean/forecast_results.csv`, `assets/charts/07_skill_forecast.png`

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from forecaster import prepare_yearly_series_from_df, forecast_skill, get_trend_direction

DATA_PATH = PROJECT_ROOT / 'data' / 'clean' / 'primary_skills_long.csv'
OUTPUT_CSV = PROJECT_ROOT / 'data' / 'clean' / 'forecast_results.csv'
OUTPUT_PNG = PROJECT_ROOT / 'assets' / 'charts' / '07_skill_forecast.png'

print('Workspace paths configured ✅')

## Phase 1 — Load and Clean Track B Data

In [ ]:
df_long = pd.read_csv(DATA_PATH)
df_track_b = df_long[df_long['posted_year'] != 'Not Specified'].copy()
df_track_b['posted_year'] = df_track_b['posted_year'].astype(int)
print(f"Loaded {len(df_track_b)} rows for Track B analysis.")

## Phase 2 — Run Tiered Forecasts

In [ ]:
top_10_skills = df_track_b['skill'].value_counts().head(10).index.tolist()
results = []

for skill in top_10_skills:
    res = forecast_skill(df_track_b, skill)
    if res:
        yearly = prepare_yearly_series_from_df(df_track_b, skill)
        res['direction'] = get_trend_direction(yearly, res['forecast_value'])
        results.append(res)

forecast_df = pd.DataFrame(results)
forecast_df

## Phase 3 — Save Forecast Output

In [ ]:
forecast_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved forecast table to {OUTPUT_CSV} ✅")

## Phase 4 — Visualizing Historical + 2027 Projections

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for skill in forecast_df['skill'].head(5):
    yearly = prepare_yearly_series_from_df(df_track_b, skill)
    ax.plot(yearly['year'], yearly['y'], marker='o', label=f"{skill} (historical)")

    fc_row = forecast_df[forecast_df['skill'] == skill].iloc[0]
    ax.plot([yearly['year'].iloc[-1], fc_row['forecast_year']],
            [yearly['y'].iloc[-1], fc_row['forecast_value']],
            linestyle='--', marker='x')

ax.axvline(2026.5, linestyle=':', color='grey', label='Forecast ->')
ax.set_title("Skill Demand: Historical (2020-2026) + 2027 Projection")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved line chart to {OUTPUT_PNG} ✅")